<a href="https://colab.research.google.com/github/chamarairesh1982/LearnPython/blob/main/chamara_of_Neural_Network_with_Keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Neural Network with Keras

This notebook uses **TensorFlow/Keras** — a standard, production-grade deep learning
library — to build, train, and tune a neural network. Rather than implementing the maths
by hand, the focus here is on the **practical skill of building a network and choosing
good hyperparameters**, which is what you'll actually do in real projects.

We'll use the same handwritten digit classification task from the lecture (Slides 20–31),
so every result connects back to concepts you've already seen: weights, epochs, learning
rate, hidden layer size, overfitting, and so on.

**Structure:**
1. Build and train a first network (the basics)
2. Hyperparameters, one at a time: hidden layer size, learning rate, epochs, batch size
3. Comparing optimizers
4. Overfitting in practice, and how to fight it (Dropout, Early Stopping)
5. Systematic hyperparameter search
6. Summary + exercises

**Colab setup:** TensorFlow, scikit-learn, and matplotlib are
pre-installed in Colab — no extra setup needed.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)


---
## 1. Building Your First Network

### The dataset
We'll use `sklearn`'s built-in handwritten digits dataset — 8×8 pixel images of digits
0–9 (1,797 samples total). An image goes in, one of 10 digit classes comes out.


In [ ]:
digits = load_digits()

# Show a few sample images, identify handwritten characters variations"
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap="gray_r")
    ax.set_title(str(digits.target[i]))
    ax.axis("off")
plt.suptitle("Sample digit images")
plt.tight_layout()
plt.show()

print("Total samples:", len(digits.images))
print("Image shape:", digits.images[0].shape, "-> flattened to", digits.data[0].shape, "input features")


In [ ]:
# Normalise pixel values to 0-1, then split into train/test sets
# digits.data contains raw pixel values. In this  dataset, each pixel is a grayscale intensity ranging from 0 to 16
# So, 17 possible intensity levels, 0 through 16.
# Dividing every pixel value by 16.0 rescales that whole range down to 0 to 1
X = digits.data / 16.0
y = digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")


### Building the network

In Keras, you build a network by **stacking layers**.

- `Input(shape=(64,))` — one input node per pixel (64 pixels = 64 inputs).
- `Dense(32, activation='relu')` — a fully-connected **hidden layer** with 32 neurons. `Dense` layers connect every neuron
  to every neuron in the previous layer.
- `Dense(10, activation='softmax')` — the **output layer**, one neuron per digit class.
  `softmax` ensures all 10 outputs sum to 1 and each is between 0 and 1 — extended to handle 10 mutually
  exclusive classes cleanly.


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(64,)),
    layers.Dense(32, activation='relu'),   # hidden layer -- 32 is a HYPERPARAMETER we chose
    layers.Dense(10, activation='softmax') # output layer -- 10 neurons, one per digit
])

model.summary()


### Compiling and training

`compile()` sets up **how** the network will learn:

- **`optimizer`** — the algorithm that performs gradient descent and applies the weight
  update (ΔW = η·gradient, Wnew = W + ΔW) automatically, epoch after epoch.
  We start with plain Stochastic Gradient Descent (SGD), the library equivalent of exactly
  what the lecture described.
- **`loss`** — how error is measured (δ = actual − desired, generalised for
  multi-class classification).
- **`metrics`** — what to report while training (accuracy, in our case).

`fit()` actually runs training — this is where every epoch, forward pass, and
backward pass happens automatically.


In [ ]:
model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.5),   # learning rate = a HYPERPARAMETER
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=50,                 # epochs = a HYPERPARAMETER
    batch_size=16,              # batch size = a HYPERPARAMETER
    validation_split=0.15,      # hold out part of the training data to monitor overfitting live
    verbose=0
)

print("Training complete.")


In [ ]:
# Plot training vs validation accuracy and loss -- the standard way to "watch" training
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='training loss')
axes[0].plot(history.history['val_loss'], label='validation loss')
axes[0].set_title('Loss over epochs')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['accuracy'], label='training accuracy')
axes[1].plot(history.history['val_accuracy'], label='validation accuracy')
axes[1].set_title('Accuracy over epochs')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal test accuracy: {test_acc:.1%}")


**What to look for:** training loss should fall steadily. If validation loss starts
*rising* while training loss keeps falling, that's overfitting in action — the
network is starting to memorise the training set rather than learn general patterns.


---
## 2. Hyperparameters, One at a Time

A **hyperparameter** is a setting *you* choose before training starts — it is never
updated by the network itself (unlike weights and biases, which backpropagation adjusts
automatically). We'll now change each major hyperparameter one at a time, holding
everything else fixed, so you can see its individual effect.

To make fair comparisons, we'll wrap model-building in a function so every run starts from
the same clean slate.


In [ ]:
def build_and_train(hidden_units=32, learning_rate=0.5, epochs=50, batch_size=16,
                     activation='relu', optimizer_name='sgd', verbose=0, seed=42):
    """Builds a fresh network with the given hyperparameters, trains it, and returns
    the training history plus test accuracy."""
    tf.random.set_seed(seed)

    model = keras.Sequential([
        layers.Input(shape=(64,)),
        layers.Dense(hidden_units, activation=activation),
        layers.Dense(10, activation='softmax')
    ])

    if optimizer_name == 'sgd':
        opt = keras.optimizers.SGD(learning_rate=learning_rate)
    elif optimizer_name == 'adam':
        opt = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        opt = keras.optimizers.RMSprop(learning_rate=learning_rate)

    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size,
                         validation_split=0.15, verbose=verbose)

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    return model, history, test_acc


### 2.1 Hidden layer size

*"The more neurons, the greater the task of calculating/learning... it can lead to
overfitting."*


In [ ]:
hidden_sizes = [4, 16, 32, 128]
results_hidden = {}

for h in hidden_sizes:
    _, hist, test_acc = build_and_train(hidden_units=h, epochs=20)
    train_acc = hist.history['accuracy'][-1]
    results_hidden[h] = (train_acc, test_acc)
    print(f"hidden_units={h:>4}  |  train acc={train_acc:.1%}  |  test acc={test_acc:.1%}")


In [ ]:
sizes = list(results_hidden.keys())
train_accs = [results_hidden[h][0] for h in sizes]
test_accs = [results_hidden[h][1] for h in sizes]

plt.figure(figsize=(6, 4))
plt.plot(sizes, train_accs, 'o-', label='train accuracy')
plt.plot(sizes, test_accs, 'o-', label='test accuracy')
plt.xlabel('Hidden units')
plt.ylabel('Accuracy')
plt.title('Effect of hidden layer size')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**What to look for:** too few hidden units (4) underfits — both train and test accuracy
are lower, since the network lacks the capacity to learn the patterns. As hidden units
increase, accuracy improves. Watch whether train accuracy starts pulling noticeably ahead
of test accuracy at the largest size — that gap is the practical signature of overfitting
risk.


### 2.2 Learning rate (Slide 17)

*"Small values = slower learning but less twitchy."*


In [ ]:
learning_rates = [0.001, 0.05, 0.5, 3.0]

plt.figure(figsize=(7, 4))
for lr in learning_rates:
    _, hist, test_acc = build_and_train(learning_rate=lr, epochs=20)
    plt.plot(hist.history['loss'], label=f'lr={lr} (test acc={test_acc:.1%})')

plt.title('Effect of learning rate on training loss')
plt.xlabel('Epoch')
plt.ylabel('Training loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**What to look for:** a tiny learning rate (0.001) barely moves in 40 epochs — far too
slow. A moderate rate descends smoothly. A large rate (3.0) may be unstable — a noisy,
jagged curve, or even loss that fails to improve.


### 2.3 Number of epochs

*"Require many epochs to successfully train the network"* — but how many is enough,
and when does more start to hurt?


In [ ]:
epoch_counts = [5, 20, 50]
results_epochs = {}

for e in epoch_counts:
    _, hist, test_acc = build_and_train(epochs=e)
    train_acc = hist.history['accuracy'][-1]
    results_epochs[e] = (train_acc, test_acc)
    print(f"epochs={e:>4}  |  train acc={train_acc:.1%}  |  test acc={test_acc:.1%}")


**What to look for:** accuracy improves with more epochs up to a point, then tends to
plateau — and sometimes test accuracy stops improving (or even dips slightly) while
training accuracy keeps climbing. That gap is overfitting from *over-training*, distinct
from overfitting caused by too many hidden units — same symptom, different cause.


### 2.4 Batch size

Batch size controls how many training examples the network looks at before each weight
update — a hyperparameter the lecture's simplified description didn't cover in detail, but
one you'll always need to set in practice.

- **Small batch size**: weight updates happen more often, using less data each time —
  noisier but often faster to start improving.
- **Large batch size**: updates are smoother (averaged over more examples) but happen less
  often per epoch.


In [ ]:
batch_sizes = [4, 16, 64, 256]

plt.figure(figsize=(7, 4))
for bs in batch_sizes:
    _, hist, test_acc = build_and_train(batch_size=bs, epochs=20)
    plt.plot(hist.history['loss'], label=f'batch_size={bs} (test acc={test_acc:.1%})')

plt.title('Effect of batch size on training loss')
plt.xlabel('Epoch')
plt.ylabel('Training loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


---
## 3. Comparing Optimizers

The lecture described plain gradient descent: compute the gradient, scale by
η, update the weights. In practice, libraries offer **smarter optimizers** that adapt the
effective learning rate automatically as training proceeds — a more sophisticated version
of the "reduce η over time" idea we discussed earlier.

- **SGD** — the vanilla approach from the lecture.
- **Adam** — adapts the learning rate per-parameter based on recent gradient history; a
  very common default in modern practice.
- **RMSprop** — similar adaptive idea, an ancestor of Adam.


In [ ]:
plt.figure(figsize=(7, 4))
for opt_name in ['sgd', 'adam', 'rmsprop']:
    _, hist, test_acc = build_and_train(optimizer_name=opt_name, learning_rate=0.01, epochs=20)
    plt.plot(hist.history['val_loss'], label=f'{opt_name} (test acc={test_acc:.1%})')

plt.title('Optimizer comparison (same learning rate, same epochs)')
plt.xlabel('Epoch')
plt.ylabel('Validation loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**What to look for:** at the *same* learning rate, Adam and RMSprop typically converge
faster and more smoothly than plain SGD — this is exactly why they're the default choice in
most modern deep learning work, even though the lecture only covered the simpler,
foundational SGD approach.


---
## 4. Overfitting in Practice, and Fighting It

Let's deliberately build an oversized network and train it too long, to see Slide 9's
overfitting warning happen clearly — then fix it with two standard library tools.


In [ ]:
# Deliberately overfit: large hidden layer, many epochs, small dataset
tf.random.set_seed(42)
overfit_model = keras.Sequential([
    layers.Input(shape=(64,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])
overfit_model.compile(optimizer=keras.optimizers.Adam(0.001),
                       loss='sparse_categorical_crossentropy', metrics=['accuracy'])

overfit_history = overfit_model.fit(X_train, y_train, epochs=60, batch_size=16,
                                     validation_split=0.15, verbose=0)

plt.figure(figsize=(7, 4))
plt.plot(overfit_history.history['loss'], label='training loss')
plt.plot(overfit_history.history['val_loss'], label='validation loss')
plt.title('Overfitting in action: training loss keeps falling, validation loss does not')
plt.xlabel('Epoch')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### Fix 1 — Dropout

`Dropout(0.3)` randomly disables 30% of neurons on each training step, forcing the network
to not over-rely on any single neuron — a widely used regularisation technique.

### Fix 2 — Early Stopping

Instead of guessing the "right" number of epochs in advance,
`EarlyStopping` monitors validation loss and automatically stops training once it stops
improving — restoring the best weights seen, rather than the final (possibly overfit) ones.


In [ ]:
tf.random.set_seed(42)
regularised_model = keras.Sequential([
    layers.Input(shape=(64,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])
regularised_model.compile(optimizer=keras.optimizers.Adam(0.001),
                           loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=8, restore_best_weights=True
)

reg_history = regularised_model.fit(
    X_train, y_train, epochs=60, batch_size=16,
    validation_split=0.15, callbacks=[early_stop], verbose=0
)

print(f"Training stopped after {len(reg_history.history['loss'])} epochs (out of 60 allowed)")

plt.figure(figsize=(7, 4))
plt.plot(reg_history.history['loss'], label='training loss')
plt.plot(reg_history.history['val_loss'], label='validation loss')
plt.title('With Dropout + Early Stopping: validation loss tracks training loss much better')
plt.xlabel('Epoch')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

_, overfit_test_acc = overfit_model.evaluate(X_test, y_test, verbose=0)
_, reg_test_acc = regularised_model.evaluate(X_test, y_test, verbose=0)
print(f"\nOverfit model (no regularisation) -> test accuracy: {overfit_test_acc:.1%}")
print(f"Regularised model (Dropout + Early Stopping) -> test accuracy: {reg_test_acc:.1%}")


**What to look for:** compare the two loss plots — the regularised version should show
validation loss tracking much closer to training loss (less divergence), and Early Stopping
should halt well before all 150 epochs complete. Final test accuracy is often similar or
better despite training for fewer effective epochs — proof that "train longer" isn't always
"train better".


---
## 5. Systematic Hyperparameter Search

Changing one hyperparameter at a time is good for building intuition, but in
practice you'll want to search several combinations at once. Here's a simple **grid
search** — trying every combination of a small set of choices and keeping the best.


In [ ]:
hidden_options = [16, 64]
lr_options = [0.001, 0.01]

grid_results = []
for h in hidden_options:
    for lr in lr_options:
        _, hist, test_acc = build_and_train(
            hidden_units=h, learning_rate=lr, optimizer_name='adam', epochs=20
        )
        grid_results.append({'hidden_units': h, 'learning_rate': lr, 'test_acc': test_acc})
        print(f"hidden_units={h:>3}, lr={lr:<6}  ->  test acc={test_acc:.1%}")

best = max(grid_results, key=lambda r: r['test_acc'])
print(f"\nBest combination: hidden_units={best['hidden_units']}, learning_rate={best['learning_rate']}  "
      f"(test acc={best['test_acc']:.1%})")


In a real project with more time and compute, you'd expand this grid (more hidden-layer
sizes, learning rates, batch sizes, optimizers) or use a dedicated tool like
`keras_tuner` or `sklearn`'s `GridSearchCV`/`RandomizedSearchCV` to automate the search —
the loop above is exactly what those tools do under the hood, just with more options and
smarter search strategies (e.g. random search, Bayesian optimisation).
